# Градусо-дні росту (GDD)

Цей блокнот завантажує дані про температуру, збережені у CSV-файлі, і аналізує їх. Він будує графік температури, показує найвищу й найнижчу температуру за кожен день і обчислює GDD.

Як користуватися цим блокнотом:

* Скопіюйте файл `temperature.csv` у ту саму папку, де лежить цей блокнот.
* Запустіть усі комірки кнопкою **▶︎ Run** вгорі. Вона запускає вибрану комірку й переходить до наступної.

У комірці нижче задайте в `base_temperature` базову температуру рослини.

In [ ]:
base_temperature = 10

Тепер потрібно завантажити CSV-файл за допомогою pandas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Читаємо CSV-файл з температурою
df = pd.read_csv('temperature.csv')

Тепер температуру можна показати на графіку.

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(df['date'], df['temperature'])
plt.xticks(rotation='vertical');

Коли дані прочитано, їх можна згрупувати за стовпцем `date` і знайти мінімальну й максимальну температуру для кожної дати.

In [ ]:
# Перетворюємо дату й час на дати, щоб згрупувати дані за датою.
# Беремо перші 10 символів (РРРР-ММ-ДД): так дані, зібрані до й після
# переходу на зимовий чи літній час (різні часові пояси), не дають помилку
df['date'] = pd.to_datetime(df['date'].str[:10]).dt.date

# Групуємо дані за датою, щоб аналізувати їх по днях
data_by_date = df.groupby('date')

# Знаходимо мінімальну й максимальну температуру для кожної дати
min_by_date = data_by_date.min()
max_by_date = data_by_date.max()

# Об'єднуємо мінімальні й максимальні температури в один DataFrame і робимо його плоским
min_max_by_date = min_by_date.join(max_by_date, on='date', lsuffix='_min', rsuffix='_max')
min_max_by_date = min_max_by_date.reset_index()

GDD можна обчислити за стандартною формулою GDD.

In [ ]:
def calculate_gdd(row):
    return ((row['temperature_max'] + row['temperature_min']) / 2) - base_temperature

# Обчислюємо GDD для кожного рядка
min_max_by_date['gdd'] = min_max_by_date.apply (lambda row: calculate_gdd(row), axis=1)

# Виводимо результати
print(min_max_by_date[['date', 'gdd']].to_string(index=False))